# Megatron-LM pipeline_parallel/p2p_communication.py 详解
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/pipeline_parallel/p2p_communication.py
- p2p_communication.py 是 Megatron-LM 流水线并行（Pipeline Parallelism）模块中的核心通信文件，
- 位于 megatron/core/pipeline_parallel/ 目录下。
- <font color='red'>它负责在流水线的不同 stage 之间进行点对点（Point-to-Point）的数据传输，是流水线并行训练的通信基础设施。</font>


## 整体架构与职责

在流水线并行中，模型按层被纵向切分到多个 GPU 上，每个 GPU 负责一段连续的层（称为一个 stage）。训练过程中:
- 前向传播的激活值需要沿流水线方向（从前一个 stage 到后一个 stage）传递，
- 反向传播的梯度则沿反方向传递。
<font color='red'>p2p_communication.py 正是封装了这些 stage 间通信的原语。</font>

该文件依赖 megatron/core/parallel_state.py 中定义的以下关键函数来获取通信拓扑信息：
- get_pipeline_model_parallel_<font color='red'>group()</font>：获取当前 rank 所在的 pipeline 并行通信组
- get_pipeline_model_parallel_<font color='red'>rank()</font>：获取当前 rank 在 pipeline 组中的序号
- get_pipeline_model_parallel_<font color='red'>prev_rank()</font>：获取前一个 stage 的 rank
- get_pipeline_model_parallel_<font color='red'>next_rank()</font>：获取后一个 stage 的 rank

## 核心底层函数：_communicate()

<font color='red'>所有高层通信 API 最终都调用 _communicate() 这个核心函数来完成实际的数据收发</font>。

其函数签名如下：

In [ ]:
def _communicate(
    *,
    tensor_send_next: Optional[torch.Tensor],  # 发送给后一个 rank 的 tensor
    tensor_send_prev: Optional[torch.Tensor],  # 发送给前一个 rank 的 tensor
    recv_prev: bool,                            # 是否从前一个 rank 接收
    recv_next: bool,                            # 是否从后一个 rank 接收
    tensor_shape: Shape,                        # 接收 tensor 的形状
    batch_p2p_comm: bool = True,                # 是否使用批量通信
    wait_on_reqs: bool = True,                  # 是否阻塞等待通信完成
    dtype: Optional[torch.dtype] = None,        # 接收 tensor 的数据类型
    variable_seq_lengths: bool = False,         # 序列长度是否可变
    use_ring_exchange_p2p: bool = False,        # 是否使用 ring_exchange
    overlap_p2p_comm: bool = False,             # 是否异步通信（用于通信重叠）
) -> Tuple[torch.Tensor, torch.Tensor]:

## 执行流程
### 初始化接收缓冲区：根据 recv_prev 和 recv_next 标志，分配用于接收数据的 CUDA 空 tensor：

In [ ]:
if recv_prev:
    tensor_recv_prev = torch.empty(
        recv_prev_shape, requires_grad=True,
        device=torch.cuda.current_device(), dtype=dtype)
if recv_next:
    tensor_recv_next = torch.empty(
        recv_next_shape, requires_grad=True,
        device=torch.cuda.current_device(), dtype=dtype)

### 选择通信后端：根据配置选择三种通信模式之一：
- <font color='red'>_batched_p2p_ops</font>：调用 torch.distributed.batch_isend_irecv()，将多个 send/recv 操作打包成一个批量请求，减少通信开销
- <font color='red'>_p2p_ops</font>：分别调用 isend() 和 irecv()，逐个发起异步通信
- <font color='red'>_ring_exchange_wrapper</font>：使用自定义的 torch.distributed.ring_exchange() 内核，适用于特定拓扑

### 执行通信并等待：

In [ ]:
reqs = p2p_func(
    tensor_send_prev=tensor_send_prev,
    tensor_recv_prev=tensor_recv_prev,
    tensor_send_next=tensor_send_next,
    tensor_recv_next=tensor_recv_next,
    group=get_pipeline_model_parallel_group())

if wait_on_reqs and len(reqs) > 0:
    for req in reqs:
        req.wait()

## 高层通信 API
基于 _communicate()，文件定义了多个语义明确的高层接口，供 schedules.py 中的调度器调用：

### 基础单向通信

| 函数 | 方向 | 用途 |
|------|------|------|
| `recv_forward()` | 从前一 stage 接收 | 前向传播时接收激活值 |
| `send_forward()` | 向后一 stage 发送 | 前向传播时发送激活值 |
| `recv_backward()` | 从后一 stage 接收 | 反向传播时接收梯度 |
| `send_backward()` | 向前一 stage 发送 | 反向传播时发送梯度 |

### 批量双向通信（用于流水线效率优化）
| 函数 | 方向 | 用途 |
|------|------|------|
| `send_forward_recv_backward()` | 向后发送 + 从后接收 | 稳态阶段同时发送前向激活和接收反向梯度 |
| `send_backward_recv_forward()` | 向前发送 + 从前接收 | 稳态阶段同时发送反向梯度和接收下一个微批次的前向输入 |
| `send_forward_recv_forward()` | 向后发送 + 从前接收 | Interleaved 流水线中使用 |
| `send_backward_recv_backward()` | 向前发送 + 从后接收 | Interleaved 流水线中使用 |
| `send_forward_backward_recv_forward_backward()` | 全双工通信 | 同时向前后方向发送和接收，用于 Interleaved 1F1B 稳态阶段 |


#### 以 send_forward_recv_backward 为例：

In [ ]:
def send_forward_recv_backward(output_tensor, tensor_shape, dtype, config):
    
    _, output_tensor_grad, _ = _communicate(
        tensor_send_next=output_tensor,    # 向后一 stage 发送前向激活
        tensor_send_prev=None,
        recv_prev=False,
        recv_next=True,                     # 从后一 stage 接收反向梯度
        tensor_shape=tensor_shape,
        config=config)
    return output_tensor_grad

## 通信优化机制
### 批量通信（Batch P2P）
当 <font color='red'>batch_p2p_comm=True</font> 时，多个 send/recv 操作被打包成一个 batch_isend_irecv 调用。这减少了 NCCL 的 kernel launch 开销，是默认开启的优化。
### 通信重叠（Overlap P2P Communication）
当 <font color='red'>overlap_p2p_comm=True </font>时，通信操作以非阻塞方式发起（wait_on_reqs=False），返回 wait_handles，允许计算与通信并行执行。这在 Interleaved 流水线的 warmup 和 steady 阶段中被广泛使用：

In [ ]:
# warmup 阶段异步发送前向结果
input_tensor, fwd_wait_handles = p2p_communication.send_forward_recv_forward(
    output_tensor, recv_prev=recv_prev, tensor_shape=tensor_shape,
    config=config, overlap_p2p_comm=True)

### Ring Exchange
当 <font color='red'>use_ring_exchange_p2p=True</font> 时，使用 torch.distributed.ring_exchange 替代标准的 send/recv。这是一种针对环形拓扑优化的通信原语，可以减少通信延迟。
### 奇偶 Rank 通信顺序优化
为了最大化通信重叠效果，Megatron 对奇数和偶数 rank 采用不同的通信操作下发顺序：
- 奇数 rank：recv_prev → send_next → recv_next → send_prev
- 偶数 rank：send_next → recv_prev → send_prev → recv_next
这种交错排列确保 send 和 recv 能正确配对，同时将 recv 的前同步操作与前一 stage 的前向计算重叠。

## 在调度器中的使用
p2p_communication.py 的接口被 schedules.py 中的三种调度策略调用：
1. Warmup 阶段：调用 recv_forward() 和 send_forward()，逐步填充流水线
2. Steady 阶段（1F1B）：调用 send_forward_recv_backward() 和 send_backward_recv_forward()，交替执行前向和反向，实现通信与计算的重叠
3. Cooldown 阶段：调用 recv_backward() 和 send_backward()，排空流水线中剩余的反向传播
- Interleaved 调度：额外使用 send_forward_backward_recv_forward_backward() 实现全双工通信，进一步减少气泡

## 总结
p2p_communication.py 本质上是一个通信抽象层，它将底层的 NCCL send/recv 原语封装成语义清晰的高层 API，并通过批量通信、异步重叠、ring exchange 等优化手段，最大限度地减少流水线 stage 间的通信开销。它是 Megatron-LM 流水线并行高效运行的基石，与 schedules.py 中的调度逻辑紧密配合，共同决定了流水线的吞吐量和效率。